# RandomForest 기반 분류 앙상블

> 이전 실험 기록: 아래 코드와 출력은 초기 RF와 기존 고정 파라미터를 사용한 7:3 분할 앙상블 실험이다. 현재는 외부 Train마다 단일 모델을 튜닝하는 `deal_model_paper_rf_ensemble.ipynb`를 사용한다. [현재 실행 순서](README.md)를 확인한다.

기존 13개 입력을 유지하고, 개별 모델의 최적 파라미터를 고정한 채 결합 방법을 비교한다.
RandomForest 자체도 여러 나무의 앙상블이며, 여기서는 서로 다른 종류의 모델까지 결합한다.

- 전처리: 원본 448건, 13개 컬럼, 행마다 Unknown 총 4개, 서로 다른 마스킹 10세트.
- Train/Test와 CV 모두 같은 원본 입력의 그룹을 분리한다. 임계값은 0.5로 고정한다.
- RF는 직전 RF 노트북, LR·ExtraTrees·CatBoost는 기존 3단계에서 고른 파라미터를 사용한다.
- 단일 모델 4개 + Soft Voting 4개 + Stacking 1개를 같은 5-Fold로 비교한다.
- 후보 선택은 CV Brier로 한다. AUC·Accuracy·Precision·Recall·F1·FP/FN도 모두 표시한다.
- TabICL, 전체 하이퍼파라미터 재탐색, 임계값 조정, 백엔드 모델 교체는 하지 않는다. 모두 CPU로 실행한다.

기존 전처리만 실행하므로 이전 RF·3단계·4단계 학습을 다시 돌릴 필요가 없다.
학습된 기존 배포 모델은 전체 데이터를 본 모델이므로 불러오지 않고, 설정만 재사용한다.

## 1. 라이브러리

In [1]:
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 1
CLASSIFICATION_THRESHOLD = 0.5
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 2. 동일한 전처리와 평가 분할

원본 CSV 경로는 전처리 노트북의 `SALESLUV_B2B_DATA_PATH` 설정을 따른다.
기본 RF와 튜닝 RF를 비교했던 것과 같은 Train/Test 및 5-Fold를 사용한다.

In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

ipython = get_ipython()
assert ipython is not None, "Jupyter 커널에서 실행해야 합니다."
# 원본 데이터 행과 전처리 전체 출력을 이 노트북에 다시 저장하지 않는다.
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

X_train_raw = ipython.user_ns["X_train_raw"]
y_train = ipython.user_ns["y_train"]
train_group_ids = ipython.user_ns["train_group_ids"]
X_test_raw_sets = ipython.user_ns["X_test_raw_sets"]
y_test = ipython.user_ns["y_test"]
input_group_ids = ipython.user_ns["input_group_ids"]
MODEL_FEATURE_NAMES = ipython.user_ns["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = ipython.user_ns["CATEGORY_VALUES"]
SOURCE_SHA256 = ipython.user_ns["SOURCE_SHA256"]

# 입력 순서, 정답 정렬, 마스킹 개수와 Train/Test 그룹 분리를 검증한다.
assert list(X_train_raw.columns) == list(MODEL_FEATURE_NAMES)
assert X_train_raw.index.equals(y_train.index)
assert len(train_group_ids) == len(y_train)
assert X_train_raw.eq("Unknown").sum(axis=1).eq(4).all()
assert set(y_train.unique()) == {0, 1}
assert len(X_test_raw_sets) == 10
for X_test_raw in X_test_raw_sets.values():
    assert list(X_test_raw.columns) == list(MODEL_FEATURE_NAMES)
    assert X_test_raw.index.equals(y_test.index)
    assert X_test_raw.eq("Unknown").sum(axis=1).eq(4).all()
    assert set(train_group_ids).isdisjoint(input_group_ids.loc[X_test_raw.index])

# 기본 모델과 튜닝 후보가 모두 같은 검증 행을 평가하도록 폴드를 한 번만 만든다.
cv5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(cv5.split(X_train_raw, y_train, groups=train_group_ids))
for train_index, valid_index in cv_splits:
    assert set(train_group_ids[train_index]).isdisjoint(set(train_group_ids[valid_index]))
    assert set(y_train.iloc[train_index].unique()) == {0, 1}
    assert set(y_train.iloc[valid_index].unique()) == {0, 1}

train_original_rows = X_train_raw.index.get_level_values("original_row_id").nunique()
print(f"Train: 원본 {train_original_rows}건 → 마스킹 포함 {len(y_train)}행")
print(f"Test: 같은 {len(y_test)}건에 서로 다른 마스킹 {len(X_test_raw_sets)}세트")
print(f"입력 {len(MODEL_FEATURE_NAMES)}개, CV {len(cv_splits)}-Fold, 임계값 0.5")

Train: 원본 313건 → 마스킹 포함 3130행
Test: 같은 135건에 서로 다른 마스킹 10세트
입력 13개, CV 5-Fold, 임계값 0.5


### 해석

Train 3,130행은 원본 거래 313건에 마스킹 10개씩을 적용한 것이다.
Test도 서로 다른 1,350건이 아니라 같은 135건을 10가지 입력 조건으로 평가한다.
따라서 마스킹 세트별 점수 차이를 독립 표본 10회의 통계적 확신으로 해석하지 않는다.

## 3. 결합할 단일 모델

### 3.1 RandomForest

기준 모델이다. 여러 나무가 입력의 비선형 관계를 학습한다.
`deal_model_random_forest.ipynb`의 CV Brier 최적 설정을 그대로 고정한다.

In [3]:
def make_one_hot_model(classifier):
    """고정된 13개 범주형 입력을 원핫으로 바꾸는 파이프라인."""
    encoder = OneHotEncoder(
        categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
        drop="first",
        handle_unknown="error",
        sparse_output=False,
        dtype=np.float32,
    )
    return Pipeline([("onehot", encoder), ("classifier", classifier)])


model_rf = make_one_hot_model(
    RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
)

### 3.2 LogisticRegression

트리와 다른 선형 판단을 더하고, 규제로 복잡한 입력 조합에 과하게 반응하는 것을 줄인다.
기존 `deal_model_phase3.ipynb`의 설정: `C=0.03, class_weight="balanced"`.

In [4]:
model_lr = make_one_hot_model(
    LogisticRegression(
        C=0.03,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )
)

### 3.3 ExtraTrees

분기 기준을 더 무작위로 선택하는 트리 모델이다. RF와 다른 예측을 더할 수 있는지 비교한다.
기존 3단계 설정: 나무 300개, 깊이 8, 잎 최소 6, `max_features="sqrt"`.

In [5]:
model_et = make_one_hot_model(
    ExtraTreesClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=6,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
)

### 3.4 CatBoost

원본 범주형 입력의 관계를 학습하는 부스팅 모델이다. 원핫 인코더 없이 같은 13개 컬럼을 받는다.
Unknown도 하나의 정상 범주로 취급한다. 기존 3단계에서 선택한 5개 파라미터를 유지한다.

In [6]:
model_catboost = CatBoostClassifier(
    iterations=200,
    depth=4,
    learning_rate=0.01,
    l2_leaf_reg=7.0,
    random_strength=0.5,
    cat_features=tuple(MODEL_FEATURE_NAMES),
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    thread_count=1,
    verbose=False,
    allow_writing_files=False,
)
base_models = {
    "RandomForest": model_rf,
    "LogisticRegression": model_lr,
    "ExtraTrees": model_et,
    "CatBoost": model_catboost,
}

## 4. Soft Voting

여러 모델의 Won 확률을 같은 비중으로 평균한다.
RF에 LR 또는 CatBoost를 하나씩 추가하고, 3개·4개 결합도 비교한다.
결합 비중은 Test를 보고 바꾸지 않는다.

In [7]:
voting_members = {
    "SoftVoting_RF_LR": ("RandomForest", "LogisticRegression"),
    "SoftVoting_RF_CatBoost": ("RandomForest", "CatBoost"),
    "SoftVoting_RF_LR_CatBoost": ("RandomForest", "LogisticRegression", "CatBoost"),
    "SoftVoting_RF_LR_ET_CatBoost": tuple(base_models),
}
voting_models = {
    name: VotingClassifier(
        estimators=[(member, base_models[member]) for member in members],
        voting="soft",
        n_jobs=1,
    )
    for name, members in voting_members.items()
}
display(
    pd.DataFrame(
        [
            {"model": name, "members": " + ".join(members)}
            for name, members in voting_members.items()
        ]
    )
)

,model,members
0,SoftVoting_RF_LR,RandomForest + LogisticRegression
1,SoftVoting_RF_CatBoost,RandomForest + CatBoost
2,SoftVoting_RF_LR_CatBoost,RandomForest + LogisticRegression + CatBoost
3,SoftVoting_RF_LR_ET_CatBoost,RandomForest + LogisticRegression + ExtraTrees...


### 해석

모델 수가 많다고 반드시 좋아지지는 않는다. 2개 조합과 4개 조합을 함께 확인한다.
최종 결과는 평균 확률이 0.5 이상이면 Won, 미만이면 Lost다.

## 5. Stacking

4개 모델의 Won 확률을 입력으로 받아 LogisticRegression이 결합 방법을 학습한다.
결합 모델에는 각 거래를 학습하지 않은 모델이 만든 예측, 즉 OOF 확률만 제공한다.

기본 `StackingClassifier(cv=3)`의 행 단위 분할을 그대로 사용하지 않는다.
마스킹 변형들이 섞이지 않도록 미리 만든 그룹 분할을 전달한다.

In [8]:
meta_splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
meta_splits = list(meta_splitter.split(X_train_raw, y_train, groups=train_group_ids))
for inner_train, inner_valid in meta_splits:
    assert set(train_group_ids[inner_train]).isdisjoint(train_group_ids[inner_valid])
    assert set(y_train.iloc[inner_train].unique()) == {0, 1}

stacking_name = "Stacking_RF_LR_ET_CatBoost"
model_stacking = StackingClassifier(
    estimators=list(base_models.items()),
    final_estimator=LogisticRegression(
        C=0.1, max_iter=3000, solver="lbfgs", random_state=RANDOM_STATE
    ),
    cv=meta_splits,
    stack_method="predict_proba",
    passthrough=False,
    n_jobs=-1,
)

## 6. Train 내부 5-Fold 비교

각 바깥 검증 폴드를 완전히 제외한 상태에서 안쪽 3-Fold로 스태킹을 학습한다.
스태킹이 학습한 단일 모델의 확률은 단일 모델·Soft Voting 비교에 재사용해 중복 학습을 줄인다.

**평가 범위:** 개별 모델의 best params는 이미 전체 Train으로 고른 고정 설정이다.
아래 CV는 그 설정들의 결합 비교이며, 하이퍼파라미터 탐색까지 모두 안쪽에서 반복한 독립 검증은 아니다.
이미 여러 실험에서 확인한 Test도 완전히 새로운 최종 검증 데이터라고 해석하지 않는다.

In [9]:
def positive_probability(estimator, X):
    won_index = list(estimator.classes_).index(1)
    return estimator.predict_proba(X)[:, won_index]


def calculate_metrics(y_true, probability):
    prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "brier": brier_score_loss(y_true, probability),
        "auc": roc_auc_score(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "tp": int(tp),
    }


cv_rows = []
base_oof = pd.DataFrame(np.nan, index=X_train_raw.index, columns=list(base_models))
cv_started = perf_counter()

In [10]:
for fold, (outer_train, outer_valid) in enumerate(cv_splits, start=1):
    X_outer = X_train_raw.iloc[outer_train]
    y_outer = y_train.iloc[outer_train]
    groups_outer = train_group_ids[outer_train]
    inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE + fold)
    inner_splits = list(inner_cv.split(X_outer, y_outer, groups=groups_outer))
    for inner_train, inner_valid in inner_splits:
        assert set(groups_outer[inner_train]).isdisjoint(groups_outer[inner_valid])
        assert set(y_outer.iloc[inner_train].unique()) == {0, 1}
    inner_coverage = np.bincount(
        np.concatenate([valid for _, valid in inner_splits]), minlength=len(X_outer)
    )
    assert (inner_coverage == 1).all()

    # 분할 위치는 현재 바깥 Train 기준이다. 전체 Train의 인덱스를 그대로 넣지 않는다.
    fitted_stack = clone(model_stacking).set_params(cv=inner_splits)
    fitted_stack.fit(X_outer, y_outer)
    X_valid = X_train_raw.iloc[outer_valid]
    y_valid = y_train.iloc[outer_valid].to_numpy()
    valid_matrix = pd.DataFrame(
        {
            name: positive_probability(estimator, X_valid)
            for name, estimator in zip(base_models, fitted_stack.estimators_, strict=True)
        }
    )
    assert np.isfinite(valid_matrix.to_numpy()).all()
    base_oof.iloc[outer_valid] = valid_matrix.to_numpy()
    probabilities = {name: valid_matrix[name].to_numpy() for name in base_models}
    for name, members in voting_members.items():
        probabilities[name] = valid_matrix[list(members)].mean(axis=1).to_numpy()
    probabilities[stacking_name] = positive_probability(
        fitted_stack.final_estimator_, valid_matrix.to_numpy()
    )
    np.testing.assert_allclose(
        probabilities[stacking_name], positive_probability(fitted_stack, X_valid), atol=1e-12
    )

    # CV에서도 마스킹 세트별로 평가해 10배 증강된 FP/FN 건수를 그대로 비교하지 않는다.
    mask_labels = X_valid.index.get_level_values("mask_set").to_numpy()
    for name, probability in probabilities.items():
        assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
        mask_metrics = []
        for mask_set in np.unique(mask_labels):
            selected = mask_labels == mask_set
            mask_metrics.append(calculate_metrics(y_valid[selected], probability[selected]))
        cv_rows.append({"model": name, "fold": fold, **pd.DataFrame(mask_metrics).mean().to_dict()})
    print(f"CV {fold}/{len(cv_splits)} 완료")

cv_seconds = perf_counter() - cv_started
cv_results = pd.DataFrame(cv_rows)
cv_comparison = cv_results.groupby("model", sort=False).mean(numeric_only=True).drop(columns="fold")
cv_comparison = cv_comparison.add_prefix("cv_")
cv_comparison["cv_brier_std"] = cv_results.groupby("model")["brier"].std(ddof=0)
cv_comparison = cv_comparison.sort_values("cv_brier", kind="stable")
selected_name = cv_comparison.index[0]
best_ensemble_name = next(name for name in cv_comparison.index if name not in base_models)
assert base_oof.notna().all().all()
assert cv_results.groupby("model").size().eq(5).all()
display(cv_comparison.round(6))
print(f"CV Brier 1위: {selected_name}")
print(f"앙상블 중 CV Brier 1위: {best_ensemble_name}")
print(f"전체 CV 학습 시간: {cv_seconds:.2f}초")

CV 1/5 완료


CV 2/5 완료


CV 3/5 완료


CV 4/5 완료


CV 5/5 완료


,cv_brier,cv_auc,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_fp,cv_fn,cv_tn,cv_tp,cv_brier_std
model,,,,,,,,,,,
CatBoost,0.194832,0.767483,0.715609,0.706967,0.790542,0.737111,11.14,6.62,19.66,25.18,0.022914
Stacking_RF_LR_ET_CatBoost,0.194962,0.771115,0.719483,0.718959,0.783504,0.737093,10.68,6.84,20.12,24.96,0.019788
SoftVoting_RF_LR_CatBoost,0.195271,0.771353,0.721460,0.711228,0.798323,0.743492,11.02,6.38,19.78,25.42,0.020653
SoftVoting_RF_CatBoost,0.195649,0.767453,0.718540,0.707074,0.795703,0.740825,11.12,6.46,19.68,25.34,0.021445
SoftVoting_RF_LR_ET_CatBoost,0.195765,0.771426,0.721758,0.709786,0.801489,0.744852,11.10,6.28,19.70,25.52,0.020773
SoftVoting_RF_LR,0.196886,0.768837,0.721778,0.713234,0.798869,0.743152,11.02,6.36,19.78,25.44,0.019666
LogisticRegression,0.197362,0.768029,0.722158,0.715312,0.797697,0.743201,10.96,6.40,19.84,25.40,0.018949
ExtraTrees,0.198232,0.766639,0.721902,0.702532,0.813086,0.748998,11.46,5.92,19.34,25.88,0.021283
RandomForest,0.198926,0.760826,0.720006,0.710346,0.797094,0.742670,11.08,6.42,19.72,25.38,0.020301


CV Brier 1위: CatBoost
앙상블 중 CV Brier 1위: Stacking_RF_LR_ET_CatBoost
전체 CV 학습 시간: 11.85초


### 해석

위 순위는 Test를 열기 전에 결정한다. 단일 모델이 이기면 앙상블을 억지로 선택하지 않는다.
CV Brier는 같은 5개 폴드 점수의 평균으로, RF 단일 노트북과 같은 기준이다.
차이가 작은 후보의 순위는 확정적인 우열로 단정하지 않는다.
FP는 실제 Lost를 Won으로 표시해 주의해야 할 거래를 놓치는 경우다. FP가 증가했는지도 함께 확인한다.

## 7. Test 10세트 비교

후보 설정과 선택 결과를 고정한 뒤 Train으로 학습하고 Test를 평가한다.
Test 결과를 보고 조합·파라미터·임계값을 다시 변경하지 않는다.

In [11]:
fit_started = perf_counter()
final_stacking = clone(model_stacking).fit(X_train_raw, y_train)
fitted_models = dict(zip(base_models, final_stacking.estimators_, strict=True))
for name, model in voting_models.items():
    fitted_models[name] = clone(model).fit(X_train_raw, y_train)
fitted_models[stacking_name] = final_stacking
final_fit_seconds = perf_counter() - fit_started

test_rows = []
for name, estimator in fitted_models.items():
    for set_name, X_test_raw in X_test_raw_sets.items():
        probability = positive_probability(estimator, X_test_raw)
        test_rows.append(
            {"model": name, "mask_set": set_name, **calculate_metrics(y_test, probability)}
        )
test_results = pd.DataFrame(test_rows)
test_means = test_results.groupby("model", sort=False).mean(numeric_only=True).add_prefix("test_")
comparison = cv_comparison.join(test_means)
assert len(comparison) == 9
assert comparison.notna().all().all()
assert test_results.groupby("model").size().eq(10).all()
display(
    comparison[
        [
            "cv_brier",
            "test_brier",
            "test_auc",
            "test_accuracy",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_fp",
            "test_fn",
        ]
    ].round(6)
)
print(f"Train 최종 후보 학습 시간: {final_fit_seconds:.2f}초")

,cv_brier,test_brier,test_auc,test_accuracy,test_precision,test_recall,test_f1,test_fp,test_fn
model,,,,,,,,,
CatBoost,0.194832,0.186809,0.790222,0.740741,0.702151,0.844118,0.766377,24.4,10.6
Stacking_RF_LR_ET_CatBoost,0.194962,0.182342,0.810031,0.732593,0.715565,0.779412,0.745831,21.1,15.0
SoftVoting_RF_LR_CatBoost,0.195271,0.182730,0.809087,0.741481,0.716822,0.805882,0.758485,21.7,13.2
SoftVoting_RF_CatBoost,0.195649,0.183368,0.802919,0.740741,0.704257,0.838235,0.765199,24.0,11.0
SoftVoting_RF_LR_ET_CatBoost,0.195765,0.182294,0.809899,0.746667,0.717635,0.820588,0.765444,22.0,12.2
SoftVoting_RF_LR,0.196886,0.182217,0.811765,0.731852,0.719328,0.769118,0.742927,20.5,15.7
LogisticRegression,0.197362,0.185205,0.806080,0.713333,0.737183,0.673529,0.703239,16.5,22.2
ExtraTrees,0.198232,0.181914,0.812028,0.742963,0.705081,0.842647,0.767612,24.0,10.7
RandomForest,0.198926,0.182146,0.804500,0.735556,0.705184,0.817647,0.756943,23.3,12.4


Train 최종 후보 학습 시간: 4.44초


### 해석

Test 지표는 동일한 135건의 마스킹 10세트 평균이다. FP/FN이 소수로 표시되는 이유도 세트 평균이기 때문이다.
Brier만 개선되고 AUC·Accuracy 또는 FP가 나빠지는 경우를 숨기지 않는다.
표의 순서는 Test 성능이 아닌 앞서 정한 CV Brier 순서다. 논문 모델을 재현한 표가 아니다.

## 8. 앙상블 개선 폭과 단건 예측 시간

In [12]:
baseline_test = test_means.loc["RandomForest"]
delta = test_means.subtract(baseline_test).loc[list(voting_models) + [stacking_name]]
display(delta[["test_brier", "test_auc", "test_accuracy", "test_fp", "test_fn"]].round(6))
display(base_oof.corr().round(3))

known_row = {
    column: next(value for value in CATEGORY_VALUES[column] if value != "Unknown")
    for column in MODEL_FEATURE_NAMES
}
masked_row = {**known_row, **dict.fromkeys(MODEL_FEATURE_NAMES[:4], "Unknown")}
self_check_X = pd.DataFrame(
    [known_row, masked_row, dict.fromkeys(MODEL_FEATURE_NAMES, "Unknown")],
    columns=list(MODEL_FEATURE_NAMES),
)
timing_rows = []
for name, estimator in fitted_models.items():
    X_timing = self_check_X.iloc[[1]]
    estimator.predict_proba(X_timing)
    durations = []
    for _ in range(20):
        started = perf_counter()
        estimator.predict_proba(X_timing)
        durations.append((perf_counter() - started) * 1000)
    timing_rows.append(
        {
            "model": name,
            "warm_predict_mean_ms": np.mean(durations),
            "warm_predict_p95_ms": np.percentile(durations, 95),
        }
    )
timing_comparison = pd.DataFrame(timing_rows).set_index("model")
display(timing_comparison.round(3))

,test_brier,test_auc,test_accuracy,test_fp,test_fn
model,,,,,
SoftVoting_RF_LR,0.000070,0.007265,-0.003704,-2.8,3.3
SoftVoting_RF_CatBoost,0.001222,-0.001580,0.005185,0.7,-1.4
SoftVoting_RF_LR_CatBoost,0.000584,0.004587,0.005926,-1.6,0.8
SoftVoting_RF_LR_ET_CatBoost,0.000148,0.005399,0.011111,-1.3,-0.2
Stacking_RF_LR_ET_CatBoost,0.000196,0.005531,-0.002963,-2.2,2.6


,RandomForest,LogisticRegression,ExtraTrees,CatBoost
RandomForest,1.000,0.955,0.991,0.955
LogisticRegression,0.955,1.000,0.960,0.941
ExtraTrees,0.991,0.960,1.000,0.960
CatBoost,0.955,0.941,0.960,1.000


,warm_predict_mean_ms,warm_predict_p95_ms
model,,
RandomForest,5.957,6.396
LogisticRegression,0.880,0.909
ExtraTrees,5.712,6.253
CatBoost,0.104,0.125
SoftVoting_RF_LR,6.194,6.379
SoftVoting_RF_CatBoost,5.386,5.511
SoftVoting_RF_LR_CatBoost,6.320,6.488
SoftVoting_RF_LR_ET_CatBoost,11.589,11.746
Stacking_RF_LR_ET_CatBoost,11.789,12.190


### 해석

개선 폭은 튜닝 RF를 뺀 값이다. AUC·Accuracy는 양수, Brier·FP/FN은 음수일 때 해당 지표가 개선된다.
OOF 확률의 상관이 높으면 서로 비슷하게 판단하므로 모델을 추가해도 개선이 작을 수 있다.
예측 시간은 로컬 CPU에서 모델을 로드한 뒤 합성 입력 1건을 20회 측정한 값이며 AWS·네트워크·LLM 시간은 제외한다.

## 9. 비교 결과와 후보 저장

In [13]:
# 선택은 CV에서 끝났다. Test 순위가 다르더라도 자동으로 후보를 바꾸지 않는다.
selected_model = fitted_models[selected_name]
artifact_path = (
    preprocessing_notebook.parents[2]
    / "backend"
    / "pipeline"
    / "artifacts"
    / "deal-rf-ensemble-v1.joblib"
)
artifact_path.parent.mkdir(parents=True, exist_ok=True)
bundle = {
    "schema_version": 1,
    "model_version": "deal-rf-ensemble-v1",
    "selected_candidate": selected_name,
    "best_ensemble_candidate": best_ensemble_name,
    "model": selected_model,
    "best_ensemble_model": fitted_models[best_ensemble_name],
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "target": {"Lost": 0, "Won": 1},
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "source_sha256": SOURCE_SHA256,
    "training_scope": "train_split_only",
    "training_original_rows": train_original_rows,
    "training_masked_rows": len(y_train),
    "cv_comparison": cv_comparison,
    "test_comparison": test_means,
    "test_results": test_results,
    "prediction_timing": timing_comparison,
    "versions": {
        name: version(name) for name in ("scikit-learn", "numpy", "pandas", "joblib", "catboost")
    },
}
joblib.dump(bundle, artifact_path)
restored = joblib.load(artifact_path)
np.testing.assert_allclose(
    selected_model.predict_proba(self_check_X),
    restored["model"].predict_proba(self_check_X),
    rtol=1e-12,
    atol=1e-12,
)
assert np.isfinite(restored["model"].predict_proba(self_check_X)).all()
assert set(restored["model"].classes_) == {0, 1}
np.testing.assert_allclose(
    fitted_models[best_ensemble_name].predict_proba(self_check_X),
    restored["best_ensemble_model"].predict_proba(self_check_X),
    rtol=1e-12,
    atol=1e-12,
)

# 실행한 sklearn VotingClassifier의 확률이 평가에 사용한 단순 평균과 같은지도 확인한다.
for name, members in voting_members.items():
    expected = np.mean(
        [positive_probability(fitted_models[member], self_check_X) for member in members], axis=0
    )
    np.testing.assert_allclose(
        positive_probability(fitted_models[name], self_check_X), expected, atol=1e-12
    )
print(f"저장 후보: {selected_name}")
print(f"함께 저장한 앙상블 후보: {best_ensemble_name}")
print(f"후보 파일: backend/pipeline/artifacts/{artifact_path.name}")
print(f"파일 크기: {artifact_path.stat().st_size / 1024**2:.3f} MiB")
print("그룹 분리·9개 후보 비교·Voting 평균·저장 후 재로드 검증: 통과")

저장 후보: CatBoost
함께 저장한 앙상블 후보: Stacking_RF_LR_ET_CatBoost
후보 파일: backend/pipeline/artifacts/deal-rf-ensemble-v1.joblib
파일 크기: 11.540 MiB
그룹 분리·9개 후보 비교·Voting 평균·저장 후 재로드 검증: 통과


### 해석

전체 CV 1위 모델, 앙상블 중 CV 1위 모델과 집계 결과를 같은 후보 파일에 저장한다. 기존 RF 후보·Stacking 배포 파일·백엔드 코드는 교체하지 않는다.
후보는 Test를 제외한 Train으로 학습되어 있다. 원본 영업 행과 모델 산출물은 Git에 올리지 않는다.

방법 참고: [scikit-learn Voting](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier),
[StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html).